In [30]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [33]:
from src.trainer.SimpleTrainer import SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils.general import InContextHead
from src.models import get_mnist_model
from src.regulariser import MultiRegulariser, L2Regulariser, UnbiasRegulariser

### Class Incremental Learning

In [34]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = get_mnist_model(device="cuda")

In [35]:
print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

trainer = SimpleTrainer(model, paradigm="CIL")
for i, (train, val, _) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    print(f"Training on task: {i + 1}")
    trainer.train(train, val, epochs=3, batch_size=256)
    trainer.test(test_tasks[0 : i + 1])

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]
Training on task: 1


Training Epochs: 100%|██████████████████| 3/3 [00:02<00:00,  1.16it/s, train_loss=0.0103, val_loss=0.0416, val_acc=0.988]


Test Results: [(0.0205, 0.9935)]
Training on task: 2


Training Epochs: 100%|████████████████████| 3/3 [00:02<00:00,  1.16it/s, train_loss=0.7015, val_loss=1.01, val_acc=0.737]


Test Results: [(1.8611, 0.0), (0.7015, 0.8684)]
Training on task: 3


...

Test Results: [(2.3244, 0.0), (2.1986, 0.0), (2.1903, 0.4824)]
Training on task: 4


Training Epochs: 100%|████████████████████████| 3/3 [00:02<00:00,  1.20it/s, train_loss=2.2323, val_loss=2.28, val_acc=0]


Test Results: [(2.3667, 0.0), (2.241, 0.0), (2.2326, 0.4824), (2.2301, 0.0)]
Training on task: 5


Training Epochs: 100%|████████████████████████| 3/3 [00:02<00:00,  1.07it/s, train_loss=2.2765, val_loss=2.34, val_acc=0]


Test Results: [(2.4113, 0.0), (2.2856, 0.0), (2.2772, 0.0), (2.2747, 0.0), (2.2758, 0.4762)]


### Task Incremental Learning

In [71]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = get_mnist_model(head=head, device="cuda")

trainer = SimpleTrainer(model, paradigm="TIL")
for i, (train, val, _) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    print(f"Training on task: {i + 1}")
    trainer.train(train, val, epochs=3, batch_size=256, context_id=i)
    trainer.test(test_tasks[0 : i + 1], context_list=list(range(0, i + 1)))

Training on task: 1


Training Epochs: 100%|████████████████████████████████████████| 3/3 [00:19<00:00,  6.40s/it, val_loss=0.0369, val_acc=0.991]


Test Results: [(0.0209, 0.993)]
Training on task: 2


Training Epochs: 100%|█████████████████████████████████████████| 3/3 [00:18<00:00,  6.15s/it, val_loss=0.102, val_acc=0.971]


Test Results: [(0.0351, 0.993), (0.0795, 0.9713)]
Training on task: 3


...

Test Results: [(0.0495, 0.9935), (0.0867, 0.9753), (0.0163, 0.994)]
Training on task: 4


Training Epochs: 100%|████████████████████████████████████████| 3/3 [00:17<00:00,  5.67s/it, val_loss=0.0776, val_acc=0.973]


Test Results: [(0.0559, 0.9925), (0.0904, 0.9723), (0.0131, 0.9935), (0.0513, 0.9802)]
Training on task: 5


Training Epochs: 100%|████████████████████████████████████████| 3/3 [00:22<00:00,  7.36s/it, val_loss=0.0446, val_acc=0.988]


Test Results: [(0.0608, 0.992), (0.0876, 0.9743), (0.0128, 0.995), (0.0537, 0.9781), (0.0402, 0.9935)]


### Domain Incremental Learning

In [72]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [75]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = get_mnist_model(device="cuda")

trainer = SimpleTrainer(model, paradigm="DIL", domain_map_fn=domain_map_fn)
for i, (train, val, _) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    print(f"Training on task: {i + 1}")
    trainer.train(train, val, epochs=3, batch_size=256)
    trainer.test(test_tasks[0 : i + 1])

Training on task: 1


Training Epochs: 100%|████████████████████████████████████████| 3/3 [00:18<00:00,  6.01s/it, val_loss=0.0352, val_acc=0.991]


Test Results: [(0.0208, 0.994)]
Training on task: 2


Training Epochs: 100%|█████████████████████████████████████████| 3/3 [00:17<00:00,  5.75s/it, val_loss=0.131, val_acc=0.964]


Test Results: [(3.4884, 0.1531), (0.1281, 0.9581)]
Training on task: 3


...

Test Results: [(2.3429, 0.5437), (1.7716, 0.6798), (0.0079, 0.9975)]
Training on task: 4


Training Epochs: 100%|█████████████████████████████████████████| 3/3 [00:16<00:00,  5.49s/it, val_loss=0.0869, val_acc=0.97]


Test Results: [(1.9781, 0.5387), (2.3947, 0.5073), (0.5955, 0.8036), (0.057, 0.9802)]
Training on task: 5


Training Epochs: 100%|████████████████████████████████████████| 3/3 [00:19<00:00,  6.52s/it, val_loss=0.0489, val_acc=0.983]


Test Results: [(3.542, 0.4859), (3.0325, 0.4644), (2.0465, 0.5937), (2.0448, 0.7009), (0.0419, 0.9908)]


### Class Incremental Learning + Regulariser

In [76]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()
model = get_mnist_model(device="cuda")

l2 = L2Regulariser(lmbd=0.01)
unbias = UnbiasRegulariser(lmbd=0.01)
multi = MultiRegulariser([l2, unbias])

trainer = SimpleTrainer(model, paradigm="CIL")
for i, (train, val, _) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    print(f"Training on task: {i + 1}")
    trainer.train(train, val, epochs=3, batch_size=256, regulariser=multi)
    trainer.test(test_tasks[0 : i + 1], regulariser=multi)

Training on task: 1


Training Epochs: 100%|██████████████████████████████████████████| 3/3 [00:23<00:00,  7.92s/it, val_loss=2.29, val_acc=0.991]


Test Results: [(0.0198, 0.9935)]
Training on task: 2


Training Epochs: 100%|██████████████████████████████████████████| 3/3 [00:22<00:00,  7.41s/it, val_loss=2.11, val_acc=0.895]


Test Results: [(3.2931, 0.0), (0.5315, 0.9203)]
Training on task: 3


...

Test Results: [(7.4029, 0.0), (3.3036, 0.0), (0.4048, 0.9884)]
Training on task: 4


Training Epochs: 100%|██████████████████████████████████████████████| 3/3 [00:22<00:00,  7.36s/it, val_loss=2.35, val_acc=0]


Test Results: [(2.4187, 0.0), (2.3611, 0.0), (2.1698, 0.5176), (2.193, 0.0)]
Training on task: 5


Training Epochs: 100%|██████████████████████████████████████████| 3/3 [00:25<00:00,  8.49s/it, val_loss=2.21, val_acc=0.891]


Test Results: [(13.3317, 0.0), (13.9757, 0.0), (13.2942, 0.0), (14.3178, 0.0), (0.4872, 0.8952)]
